# So sánh base Qwen3 với v4 trên Colab
Chọn runtime T4 GPU và chạy lần lượt các cell. Notebook dùng **base GGUF**, prompt và cấu hình phục vụ v4; đây không phải trọng số fine-tune v4. Chỉ dùng CV giả lập, không gửi CV thật qua tunnel công khai. URL công khai không có xác thực: bất kỳ ai biết URL đều có thể gọi API; không gửi CV thật. KG bật như mặc định v4, dùng seed CSV.
Cài đặt cần Internet, dung lượng cho model; fallback CPU rất chậm. Chạy cell cuối để dừng cả service lẫn tunnel.

In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = 'https://github.com/long20102004/recruit_system.git'
BRANCH = 'hocnv/DO-06'
ROOT_PREFIX = Path('/content/recruit_system_base_')
MODEL_REPO = 'unsloth/Qwen3-1.7B-GGUF'
MODEL_FILE = 'Qwen3-1.7B-Q4_K_M.gguf'
MODEL_REVISION = 'd7f544eead698dbd1f15126ef60b45a1e193322'

## Mã nguồn và phụ thuộc
Notebook không xóa hoặc sửa clone cũ `/content/recruit_system`. Clone base riêng theo 12 ký tự đầu commit branch; nếu thư mục đã tồn tại, chỉ dùng lại khi remote, HEAD khớp và không có thay đổi tracked (bỏ qua file untracked/ignored). Nếu không, dừng và báo rõ; hãy chọn prefix khác sau khi kiểm tra. Cài wheel CUDA trước; nếu không khả dụng sẽ thử CPU và cảnh báo.

In [ ]:
remote_line = subprocess.check_output(['git', 'ls-remote', REPO, 'refs/heads/' + BRANCH], text=True).strip()
if not remote_line:
    raise RuntimeError('Không tìm thấy branch public')
expected = remote_line.split()[0]
ROOT = Path(str(ROOT_PREFIX) + expected[:12])
if ROOT.exists() or ROOT.is_symlink():
    def git(*args):
        return subprocess.check_output(['git', '-C', str(ROOT), *args], text=True, stderr=subprocess.PIPE).strip()
    try:
        if git('rev-parse', '--show-toplevel') != str(ROOT) or git('remote', 'get-url', 'origin') != REPO:
            raise RuntimeError(f'{ROOT} không phải clone mong đợi; không thay đổi thư mục')
        if git('rev-parse', 'HEAD') != expected:
            raise RuntimeError(f'{ROOT} có HEAD không khớp; không thay đổi thư mục')
        if git('status', '--porcelain', '--untracked-files=no'):
            raise RuntimeError(f'{ROOT} có thay đổi tracked; không thay đổi thư mục, hãy chọn ROOT_PREFIX khác')
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(f'Không thể kiểm tra {ROOT}; không thay đổi thư mục') from exc
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)], check=True)
    actual = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != expected:
        raise RuntimeError('Branch thay đổi trong lúc clone; kiểm tra thủ công')
print('Thư mục base:', ROOT)
print('Revision:', expected)
SERVICE = ROOT / 'ai-service'
pip = [sys.executable, '-m', 'pip', 'install', '-q']
subprocess.run(pip + ['fastapi', 'uvicorn', 'pydantic', 'python-dotenv', 'json-repair', 'rapidfuzz', 'psycopg[binary]', 'neo4j>=5.20', 'huggingface_hub', 'nvidia-cuda-runtime-cu12', 'nvidia-cublas-cu12'], check=True)
cuda = subprocess.run(pip + ['--only-binary=:all:', '--extra-index-url', 'https://abetlen.github.io/llama-cpp-python/whl/cu124', 'llama-cpp-python'])
if cuda.returncode:
    print('CẢNH BÁO: không cài được wheel CUDA; thử bản CPU, inference có thể rất chậm.')
    subprocess.run(pip + ['llama-cpp-python'], check=True)
import glob, site
lib_dirs = sorted({d for sp in site.getsitepackages() for d in glob.glob(os.path.join(sp, 'nvidia', '*', 'lib'))})
os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_dirs + ['/usr/local/cuda/lib64', os.environ.get('LD_LIBRARY_PATH', '')])
probe = subprocess.run([sys.executable, '-c', 'import llama_cpp; print("GPU offload:", llama_cpp.llama_supports_gpu_offload())'], capture_output=True, text=True)
if probe.returncode:
    raise RuntimeError('Không import được llama_cpp: ' + probe.stderr)
print(probe.stdout.strip())
if 'False' in probe.stdout:
    print('CẢNH BÁO: llama.cpp không hỗ trợ GPU offload; đang dùng CPU.')

In [ ]:
from huggingface_hub import hf_hub_download
MODEL_PATH = Path(hf_hub_download(repo_id=MODEL_REPO, filename=MODEL_FILE, revision=MODEL_REVISION)).resolve()
with MODEL_PATH.open('rb') as model_file:
    magic = model_file.read(4)
size = MODEL_PATH.stat().st_size
if magic != b'GGUF' or size < 500_000_000:
    raise RuntimeError('GGUF không hợp lệ hoặc quá nhỏ; kiểm tra cache Hugging Face')
print('Base GGUF:', MODEL_PATH, '| bytes:', size)

## Service và tunnel
Service mới dùng cổng trống riêng; URL base mới được in bên dưới. Cloudflare Tunnel **không có xác thực**: bất kỳ ai biết URL đều có thể gọi API. Chỉ dùng dữ liệu giả lập, KHÔNG gửi CV thật. Tắt ghi nội dung CV và access log; dừng service và tunnel khi hoàn tất.


In [ ]:
import json, re, socket, tempfile, time, urllib.request
if ('srv' in globals() and srv.poll() is None) or ('tun' in globals() and tun.poll() is None):
    raise RuntimeError('Service/tunnel cũ còn chạy; chạy cell dọn dẹp trước')
work = tempfile.TemporaryDirectory(prefix='base-service-')
work_path = Path(work.name)
env = dict(os.environ, MODEL_ENGINE='llama_cpp', PROMPT_VERSION='v4', GGUF_MODEL_PATH=str(MODEL_PATH), LOG_CV_CONTENT='false', KB_DSN='', KG_ENABLED='true', NEO4J_URI='', MAX_NEW_TOKENS='2048', CONTEXT_TOKENS='8192', POSTPROCESS_ENABLED='true', CONTINUATION_ENABLED='true')
srv = tun = None
def stop_process(p):
    if p is not None and p.poll() is None:
        p.terminate()
        try:
            p.wait(timeout=15)
        except subprocess.TimeoutExpired:
            p.kill()
            p.wait()
try:
    # Giữ socket tới khi uvicorn nhận FD để không nối nhầm service cũ.
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as listener:
        listener.bind(('127.0.0.1', 0))
        listener.listen(128)
        PORT = listener.getsockname()[1]
        print('Cổng base:', PORT)
        with open(work_path / 'server.log', 'w') as server_log:
            srv = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'main:app', '--fd', str(listener.fileno()), '--no-access-log'], cwd=SERVICE, env=env, stdout=server_log, stderr=subprocess.STDOUT, pass_fds=(listener.fileno(),))
    health = None
    for _ in range(180):
        if srv.poll() is not None:
            raise RuntimeError('Service dừng; xem log: ' + str(work_path / 'server.log'))
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{PORT}/health', timeout=3) as response:
                health = json.load(response)
            break
        except (OSError, ValueError):
            time.sleep(2)
    if not health or health.get('engine') != 'llama_cpp' or health.get('prompt_version') != 'v4' or health.get('model') != str(MODEL_PATH):
        raise RuntimeError('Health không xác nhận đúng base model: ' + str(health))
    print('Health:', health)
    # Tải binary cloudflared vào thư mục tạm, không cài hoặc ghi đè file repo.
    cloudflared = work_path / 'cloudflared'
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o700)
    tunnel_log = open(work_path / 'tunnel.log', 'w')
    tun = subprocess.Popen([str(cloudflared), 'tunnel', '--url', f'http://127.0.0.1:{PORT}'], stdout=tunnel_log, stderr=subprocess.STDOUT)
    tunnel_log.close()
    url = None
    for _ in range(60):
        if tun.poll() is not None:
            raise RuntimeError('Tunnel dừng; xem log: ' + str(work_path / 'tunnel.log'))
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', (work_path / 'tunnel.log').read_text())
        if match:
            url = match.group(0)
            break
        time.sleep(2)
    if not url:
        raise RuntimeError('Không lấy được URL tunnel')
    print('URL base:', url)
except BaseException:
    stop_process(tun)
    stop_process(srv)
    raise

## So sánh cùng một CV giả
Điền URL service v4 **không xác thực** vào `V4_URL` nếu muốn so sánh. Cả hai API đều không có xác thực; chỉ gửi CV giả lập, KHÔNG gửi CV thật. Thời gian CPU có thể rất lâu; kết quả là dữ liệu tổng hợp, không phải đánh giá chất lượng tự động.


In [ ]:
V4_URL = ''  # Tùy chọn: URL gốc service v4 của bạn, không có dấu / cuối
cv = '''NGUYEN VAN SYNTHETIC
syn.dev@example.test
EXPERIENCE
Synthetic Software | Backend Developer | 2023 - 2025
Developed REST APIs with Spring Boot and PostgreSQL.
EDUCATION
Synthetic University | Computer Science | 2019 - 2023
SKILLS
Java, Spring Boot, Docker
'''
def extract(base_url):
    payload = json.dumps({'cv_text': cv}).encode()
    req = urllib.request.Request(base_url.rstrip('/') + '/api/v1/extract-cv', data=payload, headers={'Content-Type': 'application/json'}, method='POST')
    with urllib.request.urlopen(req, timeout=600) as response:
        return json.load(response)['data']
print('Base data:', json.dumps(extract(url), ensure_ascii=False, indent=2))
if V4_URL.strip():
    print('v4 data:', json.dumps(extract(V4_URL.strip()), ensure_ascii=False, indent=2))

## Dừng và dọn phiên
Chạy cell này khi hoàn tất, kể cả sau lỗi khởi động. Nếu runtime bị ngắt, Colab sẽ chấm dứt các tiến trình của phiên.

In [ ]:
for process in (globals().get('tun'), globals().get('srv')):
    stop_process(process)
if 'work' in globals():
    work.cleanup()
print('Đã dừng service và tunnel.')